In [4]:
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains.question_answering import load_qa_chain
from langchain_community.llms import LlamaCpp
import pypdf

In [5]:
docs = []
for fname in os.listdir("../docs"):
    if fname.lower().endswith(".pdf"):
        loader = PyPDFLoader(f"../docs/{fname}")
        for i, page in enumerate(loader.load(), start = 1):
            page.metadata["source"] = fname
            page.metadata["page"] = i
            docs.append(page)
print(f"Loaded {len(docs)} pages from {len(set(d.metadata['source'] for d in docs))} PDFs")

Loaded 8 pages from 1 PDFs


In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks.")

Created 19 chunks.


In [13]:
embeddings = SentenceTransformerEmbeddings(model_name = "all-MiniLM-L6-v2")
index = FAISS.from_documents(chunks, embeddings)
index.save_local("../faiss_multi_index")

In [17]:
index = FAISS.load_local(
    "../faiss_multi_index",
    embeddings,
    allow_dangerous_deserialization = True
    )
retriever = index.as_retriever(search_kwargs={"k":4})

In [18]:
llm = LlamaCpp(
    model_path = "../models/llama-2-7b-chat.Q4_K_M.gguf",
    temperature = 0.0,
    n_ctx = 2048,
    n_gou_layers = 20,
    verbose = False)
qa_chain = load_qa_chain(llm, chain_type = "stuff")

ImportError: Could not import llama-cpp-python library. Please install the llama-cpp-python library to use this embedding model: pip install llama-cpp-python

In [19]:
def answer_with_citations(question):
    docs = retriever.get_relevant_documents(question)
    answer = qa_chain.run(input_documents = docs, question = question)
    cites = []
    for d in docs:
        cites.append(f"{d.metadata['source']}, p. {d.metadata['page']}")
    cites = list(dict.fromkeys(cites))
    return answer, cites

In [20]:
question = input("Ask a question: ")
if question.lower() not in ("exit", "quit"):
    ans, sources = answer_with_citations(question)
    print("\nAnswer:\n", ans)
    for s in sources:
        print(" -", s)

Ask a question:  what is sum


/tmp/ipykernel_370616/3692525822.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(question)


NameError: name 'qa_chain' is not defined